# Evaluation 2 — Logistic Regression Development

This notebook is **Thamindu's primary algorithm contribution**. It justifies Logistic Regression, establishes one common baseline, compares the approved General/City/Resort training scopes, tunes the model with grouped inner cross-validation, creates training-only out-of-fold (OOF) evidence for the hotel-specific novelty, and freezes one Logistic Regression configuration per scope for later evaluation.

**NO OUTER-TEST MODEL PERFORMANCE IS CALCULATED IN THIS NOTEBOOK.** The outer test positions are reconstructed only to verify the approved split size; no outer-test predictor or target object is created for modelling.

## 1. Imports and shared infrastructure

The existing project modules remain authoritative. Predictor groups are partition helpers only and never enter the feature matrix. All learned preprocessing stays inside each fold-fitted pipeline.

In [1]:
from pathlib import Path
import json
import sys
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, cross_validate

root_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(root for root in root_candidates if (root / 'src' / 'modeling.py').exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import TARGET, prepare_training_table, split_features_target, create_predictor_groups
from src.modeling import (
    RANDOM_STATE, PRIMARY_SCORING, COMMON_SCORING, MODEL_SCOPES, make_inner_cv,
    create_training_scopes, summarize_training_scopes, build_model_pipeline, CV_RESULT_COLUMNS,
)

RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'hotel_bookings.csv'
CV_OUTPUT_PATH = PROJECT_ROOT / 'reports' / 'eval2' / 'logistic_regression_cv_results.csv'
OOF_OUTPUT_PATH = PROJECT_ROOT / 'reports' / 'eval2' / 'logistic_regression_oof_results.csv'
RUN_STARTED = time.perf_counter()
convergence_messages = []
print(f'Project root: {PROJECT_ROOT}')
print(f'Primary scoring metric: {PRIMARY_SCORING}')

Project root: D:\My GitHub Projects\hotel-cancellation
Primary scoring metric: f1


## 2. Reconstruct the approved outer-training population

The approved five-fold grouped splitter is recreated with seed 42 and only its first split is used. The test positions are retained solely as a length check. This notebook never predicts or calculates metrics on those positions.

In [2]:
raw_df = pd.read_csv(RAW_PATH)
training_table = prepare_training_table(raw_df)
X, y = split_features_target(training_table)
predictor_groups = create_predictor_groups(X)

outer_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_positions, test_positions = next(outer_cv.split(X, y, groups=predictor_groups))
X_train_outer = X.iloc[train_positions].copy()
y_train_outer = y.iloc[train_positions].copy()
groups_train_outer = predictor_groups.iloc[train_positions].copy()

assert len(X) == 119210
assert len(train_positions) == len(X_train_outer) == 95375
assert len(test_positions) == 23835
assert X_train_outer.shape[1] == 30
assert 'predictor_group' not in X_train_outer.columns
assert not set(groups_train_outer.unique()).intersection(predictor_groups.iloc[test_positions].unique())
display(pd.DataFrame({'measure': ['Modelling rows', 'Outer-training rows', 'Outer-test positions', 'Predictors'], 'value': [len(X), len(X_train_outer), len(test_positions), X_train_outer.shape[1]]}))

,measure,value
0,Modelling rows,119210
1,Outer-training rows,95375
2,Outer-test positions,23835
3,Predictors,30


## 3. Approved training scopes

City and Resort are subsets of the same outer-training population. They retain the exact 30-column schema, including the constant `hotel` predictor within each specialist scope.

In [3]:
scopes = create_training_scopes(X_train_outer, y_train_outer, groups_train_outer)
scope_summary = summarize_training_scopes(scopes)
assert tuple(scopes) == MODEL_SCOPES
assert scope_summary.set_index('scope')['rows'].to_dict() == {'General': 95375, 'City': 63044, 'Resort': 32331}
for scope_name in MODEL_SCOPES:
    assert scopes[scope_name]['X'].shape[1] == 30
    assert 'hotel' in scopes[scope_name]['X'].columns
    assert 'predictor_group' not in scopes[scope_name]['X'].columns
display(scope_summary)

,scope,rows,cancellation_rate,unique_predictor_groups
0,General,95375,0.370779,66831
1,City,63044,0.416915,40872
2,Resort,32331,0.280814,25959


## 4. Logistic Regression rationale and fixed baseline

Logistic Regression suits this binary classification task, estimates cancellation probabilities, provides interpretable linear coefficients, and gives a useful linear baseline against later tree and ensemble methods. Scaling numerical predictors and one-hot encoding categorical predictors are appropriate for this model. Its limitations are the linear relationship assumed in log-odds, less natural representation of nonlinear interactions, and sensitivity to multicollinearity or redundant predictors; regularization is therefore important. It is not assumed to be the best model.

The common baseline is `solver='lbfgs'`, L2 regularization, `C=1.0`, no class weighting, and `max_iter=2000`. L2 is a standard regularized baseline, `C=1.0` is sklearn's standard inverse-regularization baseline, and 2,000 iterations provide convergence allowance. Class weighting is deferred to tuning. The estimator's default classification rule is used; there is no threshold tuning.

In [4]:
BASELINE_PARAMETERS = {'solver': 'lbfgs', 'penalty': 'l2', 'C': 1.0, 'class_weight': None, 'max_iter': 2000}
BASELINE_JSON = json.dumps(BASELINE_PARAMETERS, sort_keys=True)
METRICS = ('accuracy', 'precision', 'recall', 'f1', 'roc_auc')
cv_records = []

def collect_warnings(caught, context):
    for item in caught:
        if issubclass(item.category, ConvergenceWarning):
            convergence_messages.append(f'{context}: {item.message}')

def cv_record(scope_name, stage, values, *, tuning_method=None, best_parameters=None, observation=''):
    scope = scopes[scope_name]
    record = {column: None for column in CV_RESULT_COLUMNS}
    record.update({
        'algorithm': 'Logistic Regression', 'scope': scope_name, 'stage': stage,
        'cv_strategy': 'StratifiedGroupKFold(5)', 'random_state': RANDOM_STATE,
        'parameters': BASELINE_JSON if stage == 'baseline' else json.dumps(values['parameters'], sort_keys=True),
        'training_rows': len(scope['X']), 'predictor_groups': scope['groups'].nunique(),
        'tuning_method': tuning_method, 'best_parameters': best_parameters, 'observation': observation,
    })
    for metric in METRICS:
        record[f'mean_cv_{metric}'] = float(values[f'mean_{metric}'])
        record[f'std_cv_{metric}'] = float(values[f'std_{metric}'])
    return record

baseline_summaries = {}
for scope_name in MODEL_SCOPES:
    scope = scopes[scope_name]
    estimator = LogisticRegression(**BASELINE_PARAMETERS)
    pipeline = build_model_pipeline(scope['X'], estimator)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        scores = cross_validate(pipeline, scope['X'], scope['y'], groups=scope['groups'], cv=make_inner_cv(), scoring=COMMON_SCORING, n_jobs=-1, return_train_score=False)
    collect_warnings(caught, f'baseline {scope_name}')
    summary = {}
    for metric in METRICS:
        summary[f'mean_{metric}'] = scores[f'test_{metric}'].mean()
        summary[f'std_{metric}'] = scores[f'test_{metric}'].std()
    baseline_summaries[scope_name] = summary
    cv_records.append(cv_record(scope_name, 'baseline', summary, observation='Fixed common baseline evaluated with grouped inner CV.'))

baseline_table = pd.DataFrame([{'scope': s, **baseline_summaries[s]} for s in MODEL_SCOPES])
display(baseline_table)

,scope,mean_accuracy,std_accuracy,mean_precision,std_precision,mean_recall,std_recall,mean_f1,std_f1,mean_roc_auc,std_roc_auc
0,General,0.821263,0.003520,0.802077,0.008602,0.687879,0.013902,0.740464,0.006754,0.903522,0.002969
1,City,0.815816,0.005777,0.824358,0.008171,0.709380,0.007531,0.762551,0.007460,0.898262,0.006354
2,Resort,0.842411,0.003626,0.757022,0.007899,0.646549,0.020134,0.697201,0.010803,0.913182,0.004197


## 5. Grouped GridSearchCV tuning

Tuning is separate for each scope and uses a fresh pipeline and fresh approved grouped splitter. `C` controls inverse regularization strength: smaller values impose stronger regularization and larger values weaker regularization. `class_weight` tests whether compensation for class frequencies improves cancellation-class performance. The controlled grid has 5 C values × 2 class-weight settings = 10 candidates per scope. Solver, penalty, iteration allowance, feature set, and classification threshold remain fixed. F1 is the refit metric.

In [5]:
PARAM_GRID = {'classifier__C': [0.01, 0.1, 1.0, 10.0, 100.0], 'classifier__class_weight': [None, 'balanced']}
assert np.prod([len(v) for v in PARAM_GRID.values()]) == 10
best_parameters_by_scope = {}
tuned_summaries = {}
candidate_counts = {}

for scope_name in MODEL_SCOPES:
    scope = scopes[scope_name]
    estimator = LogisticRegression(solver='lbfgs', penalty='l2', C=1.0, class_weight=None, max_iter=2000)
    pipeline = build_model_pipeline(scope['X'], estimator)
    search = GridSearchCV(estimator=pipeline, param_grid=PARAM_GRID, scoring=COMMON_SCORING, refit=PRIMARY_SCORING, cv=make_inner_cv(), n_jobs=-1, return_train_score=False)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        search.fit(scope['X'], scope['y'], groups=scope['groups'])
    collect_warnings(caught, f'tuning {scope_name}')
    best_index = search.best_index_
    candidate_counts[scope_name] = len(search.cv_results_['params'])
    assert candidate_counts[scope_name] == 10
    best_parameters_by_scope[scope_name] = dict(search.best_params_)
    summary = {'parameters': {
        'solver': 'lbfgs', 'penalty': 'l2', 'max_iter': 2000,
        'C': search.best_params_['classifier__C'], 'class_weight': search.best_params_['classifier__class_weight'],
    }}
    for metric in METRICS:
        summary[f'mean_{metric}'] = search.cv_results_[f'mean_test_{metric}'][best_index]
        summary[f'std_{metric}'] = search.cv_results_[f'std_test_{metric}'][best_index]
    assert np.isclose(summary['mean_f1'], search.best_score_)
    tuned_summaries[scope_name] = summary
    selected_baseline = search.best_params_ == {'classifier__C': 1.0, 'classifier__class_weight': None}
    observation = 'Grid search selected the baseline configuration.' if selected_baseline else 'Grid search selected a different configuration from the baseline.'
    cv_records.append(cv_record(scope_name, 'tuned', summary, tuning_method='GridSearchCV', best_parameters=json.dumps(search.best_params_, sort_keys=True), observation=observation))
    del search

tuned_table = pd.DataFrame([{'scope': s, 'candidates': candidate_counts[s], 'best_parameters': json.dumps(best_parameters_by_scope[s], sort_keys=True), **{k: v for k, v in tuned_summaries[s].items() if k != 'parameters'}} for s in MODEL_SCOPES])
display(tuned_table)

,scope,candidates,best_parameters,mean_accuracy,std_accuracy,mean_precision,std_precision,mean_recall,std_recall,mean_f1,std_f1,mean_roc_auc,std_roc_auc
0,General,10,"{""classifier__C"": 0.1, ""classifier__class_weig...",0.817626,0.002408,0.735081,0.004657,0.794627,0.013138,0.763610,0.004756,0.904447,0.002290
1,City,10,"{""classifier__C"": 1.0, ""classifier__class_weig...",0.812987,0.008511,0.775300,0.011819,0.776652,0.010877,0.775934,0.009864,0.898814,0.006130
2,Resort,10,"{""classifier__C"": 0.1, ""classifier__class_weig...",0.834184,0.000859,0.658111,0.002846,0.852519,0.015472,0.742715,0.004242,0.913618,0.003891


## 6. Baseline versus tuned validation performance

Changes are interpreted within each scope only. They are descriptive grouped-CV differences, not claims of statistical significance, and cross-population scores are not used to claim specialist benefit.

In [6]:
comparison_rows = []
for scope_name in MODEL_SCOPES:
    base, tuned = baseline_summaries[scope_name], tuned_summaries[scope_name]
    comparison_rows.append({
        'scope': scope_name, 'baseline_f1': base['mean_f1'], 'tuned_f1': tuned['mean_f1'], 'f1_change': tuned['mean_f1'] - base['mean_f1'],
        'baseline_recall': base['mean_recall'], 'tuned_recall': tuned['mean_recall'], 'recall_change': tuned['mean_recall'] - base['mean_recall'],
        'baseline_roc_auc': base['mean_roc_auc'], 'tuned_roc_auc': tuned['mean_roc_auc'],
    })
comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table)

cv_results_df = pd.DataFrame(cv_records, columns=CV_RESULT_COLUMNS)
assert len(cv_results_df) == 6
assert tuple(cv_results_df.columns) == CV_RESULT_COLUMNS
assert set(cv_results_df['stage']) == {'baseline', 'tuned'}
assert set(cv_results_df['scope']) == set(MODEL_SCOPES)
CV_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
cv_results_df.to_csv(CV_OUTPUT_PATH, index=False)
print(f'Saved {len(cv_results_df)} CV records to {CV_OUTPUT_PATH}')

,scope,baseline_f1,tuned_f1,f1_change,baseline_recall,tuned_recall,recall_change,baseline_roc_auc,tuned_roc_auc
0,General,0.740464,0.763610,0.023146,0.687879,0.794627,0.106748,0.903522,0.904447
1,City,0.762551,0.775934,0.013383,0.709380,0.776652,0.067272,0.898262,0.898814
2,Resort,0.697201,0.742715,0.045514,0.646549,0.852519,0.205969,0.913182,0.913618


Saved 6 CV records to D:\My GitHub Projects\hotel-cancellation\reports\eval2\logistic_regression_cv_results.csv


## 7. Training-only grouped OOF predictions

Only the selected parameter dictionaries are carried forward. A new complete pipeline is cloned and fitted on each fold's training rows, then predicts only that fold's validation rows. Probabilities—not hard predictions—are used for ROC-AUC. Assertions enforce exactly one prediction per row and zero group overlap in every fold.

In [7]:
def grouped_oof_predictions(X_scope, y_scope, groups_scope, logistic_parameters, scope_name):
    estimator = LogisticRegression(
        solver='lbfgs', penalty='l2', max_iter=2000,
        C=logistic_parameters['classifier__C'],
        class_weight=logistic_parameters['classifier__class_weight'],
    )
    base_pipeline = build_model_pipeline(X_scope, estimator)
    hard = np.full(len(X_scope), -1, dtype=int)
    score = np.full(len(X_scope), np.nan, dtype=float)
    validation_count = np.zeros(len(X_scope), dtype=int)
    fold_overlaps = []
    for fold_number, (train_idx, valid_idx) in enumerate(make_inner_cv().split(X_scope, y_scope, groups=groups_scope), start=1):
        overlap = set(groups_scope.iloc[train_idx]).intersection(groups_scope.iloc[valid_idx])
        fold_overlaps.append(len(overlap))
        assert not overlap, f'{scope_name} fold {fold_number} has group overlap'
        model = clone(base_pipeline)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always', ConvergenceWarning)
            model.fit(X_scope.iloc[train_idx], y_scope.iloc[train_idx])
        collect_warnings(caught, f'OOF {scope_name} fold {fold_number}')
        hard[valid_idx] = model.predict(X_scope.iloc[valid_idx])
        score[valid_idx] = model.predict_proba(X_scope.iloc[valid_idx])[:, 1]
        validation_count[valid_idx] += 1
    assert np.all(validation_count == 1)
    assert np.all(hard >= 0)
    assert np.isfinite(score).all()
    assert max(fold_overlaps) == 0
    return hard, score, validation_count, fold_overlaps

oof_predictions = {}
for scope_name in MODEL_SCOPES:
    scope = scopes[scope_name]
    oof_predictions[scope_name] = grouped_oof_predictions(scope['X'], scope['y'], scope['groups'], best_parameters_by_scope[scope_name], scope_name)
    print(f'{scope_name}: {len(scope["X"]):,} rows predicted exactly once; fold group overlaps = {oof_predictions[scope_name][3]}')

General: 95,375 rows predicted exactly once; fold group overlaps = [0, 0, 0, 0, 0]


City: 63,044 rows predicted exactly once; fold group overlaps = [0, 0, 0, 0, 0]


Resort: 32,331 rows predicted exactly once; fold group overlaps = [0, 0, 0, 0, 0]


In [8]:
def metric_record(scope_name, evaluation_population, y_true, hard, score):
    return {
        'algorithm': 'Logistic Regression', 'scope': scope_name, 'evaluation_population': evaluation_population,
        'stage': 'tuned_oof', 'rows': len(y_true),
        'accuracy': accuracy_score(y_true, hard),
        'precision': precision_score(y_true, hard, pos_label=1, zero_division=0),
        'recall': recall_score(y_true, hard, pos_label=1, zero_division=0),
        'f1': f1_score(y_true, hard, pos_label=1, zero_division=0),
        'roc_auc': roc_auc_score(y_true, score),
        'parameters': json.dumps(best_parameters_by_scope[scope_name], sort_keys=True),
    }

general = scopes['General']
general_hard, general_score = oof_predictions['General'][:2]
city_mask = general['X']['hotel'].eq('City Hotel').to_numpy()
resort_mask = general['X']['hotel'].eq('Resort Hotel').to_numpy()
oof_records = [
    metric_record('General', 'Overall', general['y'], general_hard, general_score),
    metric_record('General', 'City', general['y'].iloc[np.flatnonzero(city_mask)], general_hard[city_mask], general_score[city_mask]),
    metric_record('General', 'Resort', general['y'].iloc[np.flatnonzero(resort_mask)], general_hard[resort_mask], general_score[resort_mask]),
    metric_record('City', 'City', scopes['City']['y'], oof_predictions['City'][0], oof_predictions['City'][1]),
    metric_record('Resort', 'Resort', scopes['Resort']['y'], oof_predictions['Resort'][0], oof_predictions['Resort'][1]),
]
oof_results_df = pd.DataFrame(oof_records)
assert len(oof_results_df) == 5
assert ((oof_results_df['scope'] == 'General') & (oof_results_df['evaluation_population'] == 'City')).any()
assert ((oof_results_df['scope'] == 'City') & (oof_results_df['evaluation_population'] == 'City')).any()
assert ((oof_results_df['scope'] == 'General') & (oof_results_df['evaluation_population'] == 'Resort')).any()
assert ((oof_results_df['scope'] == 'Resort') & (oof_results_df['evaluation_population'] == 'Resort')).any()
oof_results_df.to_csv(OOF_OUTPUT_PATH, index=False)
display(oof_results_df)
print(f'Saved {len(oof_results_df)} OOF records to {OOF_OUTPUT_PATH}')

,algorithm,scope,evaluation_population,stage,rows,accuracy,precision,recall,f1,roc_auc,parameters
0,Logistic Regression,General,Overall,tuned_oof,95375,0.817625,0.735006,0.794616,0.763649,0.904352,"{""classifier__C"": 0.1, ""classifier__class_weig..."
1,Logistic Regression,General,City,tuned_oof,63044,0.805977,0.745922,0.810797,0.777008,0.899385,"{""classifier__C"": 0.1, ""classifier__class_weig..."
2,Logistic Regression,General,Resort,tuned_oof,32331,0.840339,0.702722,0.747770,0.724546,0.908639,"{""classifier__C"": 0.1, ""classifier__class_weig..."
3,Logistic Regression,City,City,tuned_oof,63044,0.812988,0.775216,0.776632,0.775924,0.898737,"{""classifier__C"": 1.0, ""classifier__class_weig..."
4,Logistic Regression,Resort,Resort,tuned_oof,32331,0.834184,0.658051,0.852517,0.742767,0.913472,"{""classifier__C"": 0.1, ""classifier__class_weig..."


Saved 5 OOF records to D:\My GitHub Projects\hotel-cancellation\reports\eval2\logistic_regression_oof_results.csv


## 8. Training-only novelty comparison

Specialist evidence is evaluated only against the General model on the same hotel population. Positive differences mean the specialist produced a higher OOF metric; these are descriptive training-only results and do not establish final superiority.

In [9]:
novelty_rows = []
for hotel_scope in ('City', 'Resort'):
    general_row = oof_results_df.query("scope == 'General' and evaluation_population == @hotel_scope").iloc[0]
    specialist_row = oof_results_df.query('scope == @hotel_scope and evaluation_population == @hotel_scope').iloc[0]
    novelty_rows.append({
        'population': hotel_scope,
        'general_f1': general_row['f1'], 'specialist_f1': specialist_row['f1'], 'specialist_minus_general_f1': specialist_row['f1'] - general_row['f1'],
        'general_recall': general_row['recall'], 'specialist_recall': specialist_row['recall'], 'specialist_minus_general_recall': specialist_row['recall'] - general_row['recall'],
        'general_roc_auc': general_row['roc_auc'], 'specialist_roc_auc': specialist_row['roc_auc'], 'specialist_minus_general_roc_auc': specialist_row['roc_auc'] - general_row['roc_auc'],
    })
novelty_comparison = pd.DataFrame(novelty_rows)
display(novelty_comparison)

,population,general_f1,specialist_f1,specialist_minus_general_f1,general_recall,specialist_recall,specialist_minus_general_recall,general_roc_auc,specialist_roc_auc,specialist_minus_general_roc_auc
0,City,0.777008,0.775924,-0.001084,0.810797,0.776632,-0.034165,0.899385,0.898737,-0.000647
1,Resort,0.724546,0.742767,0.018220,0.747770,0.852517,0.104747,0.908639,0.913472,0.004833


## Logistic Regression Development Findings

The executed tables above provide the evidence for baseline behaviour, selected scope-specific hyperparameters, and the size of tuning changes. The novelty table compares General and specialist models only on matching City or Resort training populations using grouped OOF predictions. Any higher or lower specialist metric is therefore an observed training-only OOF difference, not proof of final superiority.

Logistic Regression remains limited by its linear log-odds form and may miss complex nonlinear interactions; correlated or redundant predictors can also affect coefficients despite L2 regularization. No other algorithm is compared here, no threshold is tuned, and no outer-test performance is inspected.

**The Logistic Regression configurations are now frozen for later final outer-holdout evaluation and cross-algorithm comparison.**

In [10]:
elapsed_minutes = (time.perf_counter() - RUN_STARTED) / 60
print(f'Notebook modelling runtime: {elapsed_minutes:.2f} minutes')
if convergence_messages:
    print('Convergence warnings occurred and require review:')
    for message in convergence_messages:
        print('-', message)
else:
    print('No Logistic Regression convergence warnings occurred.')
print('Verification complete: 119,210 modelling rows; 95,375 outer-training rows; 23,835 outer-test positions.')
print('No outer-test model prediction or metric was calculated.')

Notebook modelling runtime: 4.15 minutes
No Logistic Regression convergence warnings occurred.
Verification complete: 119,210 modelling rows; 95,375 outer-training rows; 23,835 outer-test positions.
No outer-test model prediction or metric was calculated.
